Step 3: Econometric Analysis & Visualization

Paper: Behavioral Transparency in Multi-Agent RL: Strategic Decoupling and the Economic Optimization of Tactical Shocks

Author: Kenny Ching

Affiliation: School of Business, University of Auckland

Date: March 2026

Description:
This notebook executes the statistical testing and generates the figures and tables presented in the manuscript. It evaluates the macroeconomic trajectories of artificial and biological agents following localized tactical shocks. By applying Nearest-Neighbor state-space matching, it stratifies the human baseline (Suboptimal Losers, Competent Winners, and Matched Champions) to empirically demonstrate "Convergent Rationality" and "Strategic Decoupling."

Reproducibility Targets:

    Tables 1 & 2: OLS Regressions for Strategic Yield (ΔGt​) following Negative and Positive Tactical Shocks across the three biological baselines.

    Figures 1 & 3: Distributional Shifts (KDE Plots) showing the Truncation of Downside Risk and Maximization of Upside Yield.

    Figures 2 & 4: Conversion Efficiency (Bar Charts) detailing the discrete probability of achieving a net macroeconomic surplus.

    Figure 5: The Econometric Convergence Map (Coefficient Plot) tracking the temporal decay of the AI's marginal advantage (βAI​) under strict biological controls.

Input:

    data/processed/tf_events_master_unified.csv

Dependencies:

    pandas, numpy, statsmodels, scikit-learn (for NearestNeighbors matching), seaborn, matplotlib.

In [1]:
import os
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive

# ==========================================
# 1. SETUP & DATA LOADING
# ==========================================
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

DATA_DIR = "/content/drive/My Drive/OpenAI_Data/data"
SAVE_PATH = os.path.join(DATA_DIR, "tf_events_master_unified.csv")
HORIZONS = [30, 60, 120, 180]

def run_dense_analysis_with_tables():
    if not os.path.exists(SAVE_PATH):
        print(f"Error: {SAVE_PATH} missing. Run Prep script first.")
        return

    df = pd.read_csv(SAVE_PATH)
    print(f"Loaded {len(df)} discrete tactical shocks.")

    sns.set_style("whitegrid")
    PALETTE = {
        '1. Human Losers': '#7f7f7f',
        '2. Human Winners': '#aec7e8',
        '3. Matched Winners': '#1f77b4',
        '4. OpenAI Five': '#d62728'
    }

    coef_data = []

    for shock in ['negative', 'positive']:
        print(f"\n\n{'='*100}")
        print(f"TABLE: Strategic Yield ($\Delta G_t$) following a {shock.upper()} Tactical Shock")
        print(f"{'='*100}")
        print(f"{'Horizon':<8} | {'(1) AI vs Human Losers':<25} | {'(2) AI vs Human Winners':<25} | {'(3) AI vs Matched Winners'}")
        print("-" * 100)

        sdf = df[df['shock_type'] == shock].copy()

        # Build Cohorts
        ai_ev = sdf[sdf['is_ai'] == 1].copy()
        hu_losers = sdf[(sdf['is_ai'] == 0) & (sdf['did_win'] == 0)].copy()
        hu_winners = sdf[(sdf['is_ai'] == 0) & (sdf['did_win'] == 1)].copy()

        # State-Space Matching
        if len(ai_ev) > 0 and len(hu_winners) > 0:
            scaler = hu_winners[['fight_start', 'current_gold_lead']].std()
            if scaler['fight_start'] == 0: scaler['fight_start'] = 1
            if scaler['current_gold_lead'] == 0: scaler['current_gold_lead'] = 1

            nbrs = NearestNeighbors(n_neighbors=5).fit(hu_winners[['fight_start', 'current_gold_lead']] / scaler)
            indices = nbrs.kneighbors(ai_ev[['fight_start', 'current_gold_lead']] / scaler)[1]
            matched_winners = hu_winners.iloc[np.unique(indices.flatten())].copy()
        else:
            print("Insufficient data for matching.")
            continue

        # Tag for plotting
        ai_ev['Group'] = '4. OpenAI Five'
        hu_losers['Group'] = '1. Human Losers'
        hu_winners['Group'] = '2. Human Winners'
        matched_winners['Group'] = '3. Matched Winners'

        plot_df = pd.concat([hu_losers, hu_winners, matched_winners, ai_ev])

        # Regression Loops & Table Printing
        for h in HORIZONS:
            col = f'gadv_d{h}'

            # Model 1
            m1 = smf.ols(f"{col} ~ is_ai", data=pd.concat([ai_ev, hu_losers])).fit()
            c1, p1 = m1.params['is_ai'], m1.pvalues['is_ai']
            s1 = "*" if p1 < 0.05 else "ns"
            str1 = f"{c1:>7.1f} (p={p1:.3f}) {s1}"
            coef_data.append({'Shock': shock, 'Horizon': h, 'Model': 'vs Losers', 'Coef': c1, 'SE': m1.bse['is_ai']})

            # Model 2
            m2 = smf.ols(f"{col} ~ is_ai", data=pd.concat([ai_ev, hu_winners])).fit()
            c2, p2 = m2.params['is_ai'], m2.pvalues['is_ai']
            s2 = "*" if p2 < 0.05 else "ns"
            str2 = f"{c2:>7.1f} (p={p2:.3f}) {s2}"
            coef_data.append({'Shock': shock, 'Horizon': h, 'Model': 'vs Winners', 'Coef': c2, 'SE': m2.bse['is_ai']})

            # Model 3
            m3 = smf.ols(f"{col} ~ is_ai", data=pd.concat([ai_ev, matched_winners])).fit()
            c3, p3 = m3.params['is_ai'], m3.pvalues['is_ai']
            s3 = "*" if p3 < 0.05 else "ns"
            str3 = f"{c3:>7.1f} (p={p3:.3f}) {s3}"
            coef_data.append({'Shock': shock, 'Horizon': h, 'Model': 'vs Matched', 'Coef': c3, 'SE': m3.bse['is_ai']})

            # Print Row
            print(f"{h}s      | {str1:<25} | {str2:<25} | {str3}")

        # ---------------------------------------------------------
        # PLOT 1: DISTRIBUTIONAL SHIFT (KDE)
        # ---------------------------------------------------------
        plt.figure(figsize=(9, 6))
        sns.kdeplot(data=plot_df, x='gadv_d180', hue='Group', fill=True, common_norm=False,
                    palette=PALETTE, alpha=0.3, linewidth=2)

        title_prefix = "Truncation of Downside Risk" if shock == 'negative' else "Maximization of Upside Yield"
        plt.title(f'{title_prefix} ({shock.title()} Shock at 180s)', fontsize=14, weight='bold')

        plt.xlabel(r'Strategic Yield ($\Delta$ Gold)')
        plt.axvline(0, color='black', ls='--')
        plt.xlim(-8000, 8000)
        plt.tight_layout()

        # Save directly to Drive
        dist_path = os.path.join(DATA_DIR, f'fig_{shock}_distribution.png')
        plt.savefig(dist_path, dpi=300)
        plt.close()

        # ---------------------------------------------------------
        # PLOT 2: CONVERSION EFFICIENCY (BAR)
        # ---------------------------------------------------------
        plot_df['Success'] = (plot_df['gadv_d180'] > 0).astype(int)
        eff = plot_df.groupby('Group')['Success'].agg(['mean', 'sem']).reset_index()

        plt.figure(figsize=(8, 6))
        plt.bar(eff['Group'], eff['mean'], yerr=eff['sem']*1.96, capsize=10,
                color=[PALETTE[g] for g in eff['Group']], alpha=0.85, edgecolor='black')
        plt.title(f'Conversion Efficiency: Probability of Net Surplus ({shock.title()} Shock)', fontsize=14, weight='bold')
        plt.ylabel(r'P($\Delta$ Gold > 0)')
        plt.axhline(0.5, color='gray', ls='--')
        plt.xticks(rotation=15)
        plt.tight_layout()

        # Save directly to Drive
        eff_path = os.path.join(DATA_DIR, f'fig_{shock}_efficiency.png')
        plt.savefig(eff_path, dpi=300)
        plt.close()

    # ---------------------------------------------------------
    # PLOT 3: CONVERGENCE MAP
    # ---------------------------------------------------------
    print("\n\nGenerating Econometric Plots...")
    coef_df = pd.DataFrame(coef_data)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
    fig.suptitle('Econometric Convergence: AI Advantage Decays with Stricter Biological Baselines', fontsize=16, weight='bold')

    model_colors = {'vs Losers': '#7f7f7f', 'vs Winners': '#aec7e8', 'vs Matched': '#1f77b4'}

    for i, shock in enumerate(['negative', 'positive']):
        ax = axes[i]
        subset = coef_df[coef_df['Shock'] == shock]

        for model in ['vs Losers', 'vs Winners', 'vs Matched']:
            m_data = subset[subset['Model'] == model]
            ax.errorbar(m_data['Horizon'], m_data['Coef'], yerr=m_data['SE']*1.96,
                        fmt='-o', label=model, color=model_colors[model],
                        capsize=5, linewidth=2.5, markersize=8)

        ax.axhline(0, color='black', ls='--', linewidth=1.5)
        ax.set_title(f'Response to {shock.title()} Shock', fontsize=14)
        ax.set_xlabel('Seconds Post-Shock', fontsize=12)

        if i == 0: ax.set_ylabel(r'AI Marginal Advantage ($\beta_{AI}$ in Gold)', fontsize=12)
        ax.set_xticks(HORIZONS)
        ax.grid(True, alpha=0.3)
        if i == 1: ax.legend(title='Biological Control Group')

    plt.tight_layout()

    # Save directly to Drive
    conv_path = os.path.join(DATA_DIR, 'fig_econometric_convergence.png')
    plt.savefig(conv_path, dpi=300)
    plt.close()

    print(f"✅ Execution Complete. All 5 figures have been saved to: {DATA_DIR}")

if __name__ == "__main__":
    run_dense_analysis_with_tables()

<>:40: SyntaxWarning: invalid escape sequence '\D'
<>:40: SyntaxWarning: invalid escape sequence '\D'
/tmp/ipykernel_351/3160022766.py:40: SyntaxWarning: invalid escape sequence '\D'
  print(f"TABLE: Strategic Yield ($\Delta G_t$) following a {shock.upper()} Tactical Shock")


Mounted at /content/drive
Loaded 9074 discrete tactical shocks.


TABLE: Strategic Yield ($\Delta G_t$) following a NEGATIVE Tactical Shock
Horizon  | (1) AI vs Human Losers    | (2) AI vs Human Winners   | (3) AI vs Matched Winners
----------------------------------------------------------------------------------------------------
30s      |   326.8 (p=0.210) ns      |   -33.9 (p=0.878) ns      |   160.6 (p=0.478) ns
60s      |   867.1 (p=0.021) *       |   313.9 (p=0.385) ns      |   561.0 (p=0.127) ns
120s      |  1008.7 (p=0.081) ns      |  -255.5 (p=0.636) ns      |   -63.4 (p=0.907) ns
180s      |  1677.2 (p=0.024) *       |  -349.1 (p=0.623) ns      |  -132.1 (p=0.823) ns


TABLE: Strategic Yield ($\Delta G_t$) following a POSITIVE Tactical Shock
Horizon  | (1) AI vs Human Losers    | (2) AI vs Human Winners   | (3) AI vs Matched Winners
----------------------------------------------------------------------------------------------------
30s      |   134.6 (p=0.435) ns      |  -2